# Create Wilhelm Sander-Stiftung Awards

Creates Wilhelm Sander-Stiftung (WSS, Munich; cancer and medical research
foundation) awards from the foundation's own funded-projects database at
https://www.wilhelm-sander-stiftung.de/forschungsprojekte/ (map + search). The
page reads the public WordPress REST namespace `/wp-json/foerderprojekte/v1/`
(`projects/by-university/{id}` for every university, plus `search` for the 9
projects with no university). **2,627 projects, 1976-2026, EUR 349.5M,
99.9% amount, 100% applicant** (local run 2026-10-01).

**Prerequisites:**
- Run `scripts/local/wilhelm_sander_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/wilhelm_sander/wilhelm_sander_projects.parquet`
  (runbook §1.4 shrink guard).

**Source quirks handled in the script:**
- The site lists 2,799 posts; 172 are exact duplicates (two consecutive post
  ids with identical title, applicant, year, amount, institute, university:
  a double import). The lower post id is kept; the other is recorded in
  `duplicate_post_ids`.
- Pre-2000 applicant names are stored in capitals; the script title-cases
  those (raw kept in `applicant_firstname_raw` / `applicant_lastname_raw`).

**Field notes:**
- `display_name`: project title. **726 projects funded 1976-1997 have no title
  in the source** (applicant, institute, year and amount only); they ship with
  `display_name` NULL rather than a made-up label (precedent: Royal Society
  grants). Titles are in the language submitted (mostly German, recent ones
  often English).
- `amount`: `funding_amount` in EUR (all years are shown in EUR on the site).
- Dates: `start_date`/`end_date` from the published funding period (~10% of
  projects, recent ones); otherwise `start_date` = 1 January of the funding
  year. `end_date` NULL when no period is published.
- `funder_scheme`: the WSS research line (`type`), e.g. "Klinische und
  klinisch orientierte Krebsforschung", "Experimentielle Krebsforschung",
  "Therapieinheit" (6 multi-year therapy-unit grants, kept as research
  funding). Scope filter: none; everything in the database is medical research.
- `lead_investigator`: the applicant; affiliation = the university/hospital
  named by the site (the institute stays in the raw table).

**`funder_award_id` (§2.1.1):** citing works quote the WSS grant number
(`2017.009.1`, ~1,800 priority-0 shells), but the site does not publish it
and project detail pages redirect to the home page. The ingest uses the
synthetic, stable key `WSS-{WordPress post id}`, so it collapses onto **0**
existing citation stubs (checked in the last verification cell).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320310707` (exactly one row in `openalex.funders.funders`; no twin
  by name or ROR)
- display_name: Wilhelm Sander-Stiftung
- ror_id / doi: from `openalex.funders.funders` (ror 02q83sc19, doi 10.13039/100008672)

**Priority:** `584` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.wilhelm_sander_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/wilhelm_sander/wilhelm_sander_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects, COUNT(DISTINCT funder_award_id) as distinct_ids
FROM openalex.awards.wilhelm_sander_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.wilhelm_sander_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.wilhelm_sander_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320310707 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320310707;

## Step 2: Create Wilhelm Sander-Stiftung Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.wilhelm_sander_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320310707  -- Wilhelm Sander-Stiftung
),

g AS (
    SELECT
        r.*,
        TRIM(r.funder_award_id) AS award_key,
        COALESCE(TRY_TO_DATE(r.start_date, 'yyyy-MM-dd'),
                 CASE WHEN TRY_CAST(r.year AS INT) IS NOT NULL THEN make_date(TRY_CAST(r.year AS INT), 1, 1) END) AS start_dt,
        TRY_TO_DATE(r.end_date, 'yyyy-MM-dd') AS end_dt,
        CASE WHEN NULLIF(TRIM(r.lead_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(r.lead_given_name), ''),
            'family_name', NULLIF(TRIM(r.lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(r.university), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead
    FROM openalex.awards.wilhelm_sander_raw r
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_key)))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.summary), '') as description,
    f.funder_id,
    g.award_key as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'EUR' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    NULLIF(TRIM(g.type), '') as funder_scheme,
    'wss_foerderprojekte' as provenance,
    g.start_dt as start_date,
    g.end_dt as end_date,
    YEAR(g.start_dt) as start_year,
    YEAR(g.end_dt) as end_year,
    g.lead as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN g.lead IS NOT NULL THEN array(g.lead) END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_key)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f
WHERE g.award_key IS NOT NULL AND g.award_key != '';

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'wss_foerderprojekte' AND priority = 584;

-- Insert into openalex_awards_raw with priority.
-- Priority 584: direct-from-funder ingest of the Wilhelm Sander-Stiftung
-- project database. Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    584 as priority
FROM openalex.awards.wilhelm_sander_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.wilhelm_sander_awards;

In [ ]:
%sql
-- §2.1.1 shape check: every funder_award_id is WSS-{post id}.
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^WSS-[0-9]+$' THEN 1 ELSE 0 END) as ok_shape,
    SUM(CASE WHEN funder_award_id NOT RLIKE '^WSS-[0-9]+$' THEN 1 ELSE 0 END) as bad_shape
FROM openalex.awards.wilhelm_sander_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.wilhelm_sander_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.wilhelm_sander_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, COUNT(display_name) as with_title, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.wilhelm_sander_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.wilhelm_sander_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.wilhelm_sander_awards
WHERE display_name IS NOT NULL
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (expect ~72% title, ~99.9% amount, 100% PI).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.wilhelm_sander_awards;

In [ ]:
%sql
-- Overlap with existing citation stubs for this funder (expect 0: synthetic key).
SELECT c.provenance, COUNT(*) as stubs, COUNT(t.id) as collapsed_onto_wss_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.wilhelm_sander_awards t ON t.id = c.id
WHERE c.funder_id = 4320310707 AND c.provenance != 'wss_foerderprojekte'
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'wss_foerderprojekte'
GROUP BY provenance, priority;